
# Paddy Yield Prediction using Gradient Boosting Regressor
# Data Loading

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# %%
# Load dataset
df = pd.read_csv("../data/paddydataset.csv")

# Show dataset info
print("Dataset shape:", df.shape)
df.head()

# Feature Selection

In [ ]:
# Define target variable
y = df["Paddy yield(in Kg)"]

# Define input features
X = df.drop(columns=["Paddy yield(in Kg)"])

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

: 

# Train-Test Split

In [ ]:
# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Data Preprocessing

In [ ]:
# Separate categorical and numerical columns
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

# Create preprocessing pipeline
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("var", VarianceThreshold(threshold=0.01))
        ]), num_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ]
)

# Model Evaluation

In [ ]:
# Make predictions
y_pred_gbr = gbr_model.predict(X_test)

# Evaluate performance
mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
r2_gbr = r2_score(y_test, y_pred_gbr)

print("Gradient Boosting Regressor Performance:")
print("MAE:", mae_gbr)
print("RMSE:", rmse_gbr)
print("R2 Score:", r2_gbr)

# The Gradient Boosting Regressor achieved strong predictive performance, showing its ability to model complex non-linear relationships in paddy yield prediction.

In [ ]:
# Visualization 1 - Actual vs Predicted Values

# %%
plt.figure(figsize=(6,4))
plt.scatter(y_test, y_pred_gbr, alpha=0.7)
plt.xlabel("Actual Paddy Yield")
plt.ylabel("Predicted Paddy Yield")
plt.title("Actual vs Predicted Paddy Yield")
plt.grid(True)
plt.show()

In [ ]:
# %% [markdown]
# ## Visualization 2 - Residual Plot

# %%
residuals = y_test - y_pred_gbr

plt.figure(figsize=(6,4))
plt.scatter(y_pred_gbr, residuals, alpha=0.7)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Paddy Yield")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.grid(True)
plt.show()

In [ ]:
# %% [markdown]
# ## Visualization 3 - Distribution of Residuals

# %%
plt.figure(figsize=(6,4))
plt.hist(residuals, bins=30)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Distribution of Residuals")
plt.grid(True)
plt.show()

In [ ]:
# %%
# Extract trained regressor from pipeline
regressor = gbr_model.named_steps["regressor"]

# Get transformed feature names
ohe = gbr_model.named_steps["prep"].named_transformers_["cat"].named_steps["encoder"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)

selected_num_cols = num_cols  # approximate names kept after variance threshold
all_feature_names = list(selected_num_cols) + list(cat_feature_names)

# Feature importances
importances = regressor.feature_importances_

# Match lengths safely
feature_importance_df = pd.DataFrame({
    "Feature": all_feature_names[:len(importances)],
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print(feature_importance_df.head(10))

In [ ]:
# %%
plt.figure(figsize=(8,5))
plt.barh(feature_importance_df["Feature"].head(10)[::-1],
         feature_importance_df["Importance"].head(10)[::-1])
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Top 10 Important Features")
plt.grid(True)
plt.show()